# 🎓 UniSense Yapay Zeka Asistanı Eğitimi (Qwen3-4B + LoRA)

**Hedef:** UniSense projesindeki **58.585 Türkçe Q/A** çiftiyle (üniversite, bölüm, taban puan, sıra, coğrafya, hesap mantığı) **Qwen3-4B-Instruct-2507** modelini ince ayarla — Gemini'siz çalışan yerel bir AI asistanı çıkar.

**Pipeline:** AFET projesi ile aynı (LoRA + 4-bit + GGUF)
- Model: `Qwen/Qwen3-4B-Instruct-2507`
- Method: QLoRA (4-bit quantization + r=16 LoRA)
- T4 GPU yeter (~10GB VRAM)
- **Süre tahmini: 7-10 saat** (2 epoch × 58.585 örnek, batch=2, grad_accum=8)
- Kaggle T4 12 saat oturum limiti içinde rahat biter

**Sonuç:** `unisense-tr-q4_k_m.gguf` (~2.5GB) — Ollama ile lokal CPU'da bile çalışır.

---

⚠️ **Önemli:**
- Kaggle stdout buffer sorunlu olabilir — `FlushCallback` her log adımında elle flush eder
- Cell sırası: 1 → 2 → 3 → 4 → 5 → 6 → 7
- Eğer 12 saat dolarsa: Cell 4'i `trainer.train(resume_from_checkpoint=True)` ile yeniden başlat (son checkpoint'ten devam eder)

### Adım 1: Bağımlılıklar

In [ ]:
!pip install -U transformers peft accelerate bitsandbytes --quiet

### Adım 2: Qwen3-4B-Instruct-2507 + 4-bit + LoRA

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import torch

MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
))
model.print_trainable_parameters()

### Adım 3: UniSense Dataset Yükle (chat template ile)

In [ ]:
from datasets import load_dataset
import os

SYSTEM_MSG = (
    "Sen UniSense — Türkiye 2025 YKS üniversite tercih asistanısın. "
    "YÖK Atlas/ÖSYM verilerine dayanarak Türkçe, kısa, madde işaretli, sayısal cevaplar ver. "
    "ASLA İngilizce kelime kullanma. Bilgi yoksa 'YÖK Atlas'tan kontrol et' de. "
    "Program kodunu [203910363] formatında yaz."
)

def formatting_prompts_func(examples):
    texts = []
    for instruction, output in zip(examples["instruction"], examples["output"]):
        messages = [
            {"role": "system", "content": SYSTEM_MSG},
            {"role": "user", "content": instruction},
            {"role": "assistant", "content": output},
        ]
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
        texts.append(text)
    return {"text": texts}

# Olası path'leri sırayla dene (upload yapısı farklı olabilir)
DATASET_PATHS = [
    "/kaggle/input/unisense-dataset/training/unisense_dataset.jsonl",   # training/ alt klasörü ile
    "/kaggle/input/unisense-dataset/unisense_dataset.jsonl",             # düz upload
    "/kaggle/input/datasets/ibrahimaskeroglu/unisense-dataset/unisense_dataset.jsonl",
]
DATASET_PATH = None
for p in DATASET_PATHS:
    if os.path.exists(p):
        DATASET_PATH = p
        break
# Fallback: /kaggle/input içinde recursive ara
if DATASET_PATH is None:
    for root, _, files in os.walk("/kaggle/input"):
        for fname in files:
            if fname == "unisense_dataset.jsonl":
                DATASET_PATH = os.path.join(root, fname)
                break
        if DATASET_PATH:
            break
assert DATASET_PATH, "unisense_dataset.jsonl Kaggle input'ta yok! 'Add Data' ile dataset ekledin mi?"
print(f"📂 Dataset: {DATASET_PATH}")
print(f"📦 Boyut: {os.path.getsize(DATASET_PATH) / 1024 / 1024:.2f} MB")

dataset = load_dataset("json", data_files=DATASET_PATH, split="train")
dataset = dataset.map(formatting_prompts_func, batched=True)
print(f"📚 Boyut: {len(dataset):,} örnek")
print("📋 Örnek:")
print(dataset[0]["text"][:600])

### Adım 4: Eğitim (completion-only masking)

In [ ]:
from transformers import Trainer, TrainingArguments, TrainerCallback
import torch, sys

tokenizer.pad_token = tokenizer.eos_token
RESPONSE_TEMPLATE = "<|im_start|>assistant\n"
RESPONSE_TEMPLATE_IDS = tokenizer.encode(RESPONSE_TEMPLATE, add_special_tokens=False)
print(f"Response template IDs: {RESPONSE_TEMPLATE_IDS}", flush=True)

def tokenize_and_mask(example):
    text = example["text"]
    encoded = tokenizer(text, truncation=True, max_length=768, padding=False)
    input_ids = encoded["input_ids"]
    labels = list(input_ids)
    rt_len = len(RESPONSE_TEMPLATE_IDS)
    start_idx = -1
    for i in range(len(input_ids) - rt_len + 1):
        if input_ids[i:i + rt_len] == RESPONSE_TEMPLATE_IDS:
            start_idx = i + rt_len
            break
    if start_idx == -1:
        labels = [-100] * len(labels)
    else:
        for i in range(start_idx):
            labels[i] = -100
    encoded["labels"] = labels
    return encoded

print("📝 Tokenize + mask (4 worker)...", flush=True)
tokenized_dataset = dataset.map(
    tokenize_and_mask,
    remove_columns=dataset.column_names,
    desc="Tokenizing",
    num_proc=4,
)
first = tokenized_dataset[0]
loss_tokens = sum(1 for l in first["labels"] if l != -100)
masked_tokens = sum(1 for l in first["labels"] if l == -100)
print(f"📊 İlk: {loss_tokens} cevap-token, {masked_tokens} prompt-token", flush=True)
if loss_tokens == 0:
    raise RuntimeError("Cevap token'ı yok! Template kontrol et.")

def pad_collator(features):
    max_len = max(len(f["input_ids"]) for f in features)
    batch = {"input_ids": [], "attention_mask": [], "labels": []}
    pad_id = tokenizer.pad_token_id
    for f in features:
        n = len(f["input_ids"])
        pad_n = max_len - n
        batch["input_ids"].append(f["input_ids"] + [pad_id] * pad_n)
        batch["attention_mask"].append([1] * n + [0] * pad_n)
        batch["labels"].append(f["labels"] + [-100] * pad_n)
    return {k: torch.tensor(v, dtype=torch.long) for k, v in batch.items()}

# Kaggle stdout buffer'ı ile mücadele — her log adımında elle flush et
class FlushCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            print(f"  step {state.global_step}/{state.max_steps} | "
                  f"loss={logs.get('loss','?')} lr={logs.get('learning_rate','?')}",
                  flush=True)
            sys.stdout.flush()
    def on_train_begin(self, args, state, control, **kwargs):
        print(f"🚀 Eğitim başladı | toplam adım: {state.max_steps} | "
              f"epoch: {args.num_train_epochs}", flush=True)
    def on_save(self, args, state, control, **kwargs):
        print(f"💾 Checkpoint kaydedildi: outputs/checkpoint-{state.global_step}", flush=True)

trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir="outputs",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        warmup_ratio=0.03,
        num_train_epochs=2,                # 4 → 2 (Kaggle 12h limit için yeterli)
        learning_rate=1e-4,
        max_grad_norm=0.3,
        fp16=True,
        logging_steps=25,                  # 5 → 25 (daha az spam, daha güvenilir)
        logging_first_step=True,
        save_steps=200,                    # 300 → 200 (daha sık checkpoint)
        save_total_limit=2,
        optim="paged_adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        report_to="none",
        gradient_checkpointing=True,
        disable_tqdm=False,                # progress bar GÖSTER
        dataloader_num_workers=2,
    ),
    train_dataset=tokenized_dataset,
    data_collator=pad_collator,
    callbacks=[FlushCallback()],
)
print("⏳ trainer.train() başlatılıyor...", flush=True); sys.stdout.flush()
trainer_stats = trainer.train()
print(f"\n✅ Eğitim tamamlandı. Final loss: {trainer_stats.training_loss:.4f}", flush=True)
print("   Hedef: 0.6-1.2 + tutarlı TR cevaplar.", flush=True)

### Adım 5: LoRA Adapter Kaydet → Kaggle Dataset

In [ ]:
import os, json, subprocess

UPLOAD_DIR = "/kaggle/working/upload-unisense-lora-4b"
os.makedirs(UPLOAD_DIR, exist_ok=True)

with open(f"{UPLOAD_DIR}/dataset-metadata.json", "w") as f:
    json.dump({
        "title": "unisense-lora-4b",
        "id": "ibrahimaskeroglu/unisense-lora-4b",
        "licenses": [{"name": "CC0-1.0"}]
    }, f)

model.save_pretrained(f"{UPLOAD_DIR}/lora_unisense_model")
tokenizer.save_pretrained(f"{UPLOAD_DIR}/lora_unisense_model")

print("⬆️ Kaggle'a yükleniyor (unisense-lora-4b)...")
result = subprocess.run(
    ["kaggle", "datasets", "create", "-p", UPLOAD_DIR, "--dir-mode", "zip"],
    capture_output=True, text=True
)
if "already exists" in (result.stdout + result.stderr).lower():
    result = subprocess.run(
        ["kaggle", "datasets", "version", "-p", UPLOAD_DIR, "-m", "UniSense Qwen3-4B TR LoRA", "--dir-mode", "zip"],
        capture_output=True, text=True
    )
print("STDOUT:", result.stdout[-500:])
print("STDERR:", result.stderr[-500:])

### Adım 6: Hızlı Doğrulama

In [ ]:
import torch

model.eval()
if hasattr(model, "gradient_checkpointing_disable"):
    model.gradient_checkpointing_disable()
if hasattr(model, "base_model") and hasattr(model.base_model, "gradient_checkpointing_disable"):
    try:
        model.base_model.gradient_checkpointing_disable()
    except Exception:
        pass
model.config.use_cache = True
print("🔧 Inference modu hazır")

test_questions = [
    "Boğaziçi Bilgisayar Mühendisliği taban puanı nedir?",
    "300.000 sıra ile devlet üniversitelerine yerleşebilir miyim?",
    "İlgi alanım: hasta bakımı, klinik. Hangi bölümleri seçebilirim?",
    "Karadeniz kıyısındaki üniversiteler hangileri?",
    "TYT puanı nasıl hesaplanır?",
    "Mersin'deki üniversiteler",
]

def generate_answer(question: str, max_new_tokens: int = 350) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_MSG},
        {"role": "user", "content": question},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer([prompt], return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs,
            max_new_tokens=max_new_tokens,
            min_new_tokens=30,
            temperature=0.7, top_p=0.95, top_k=50,
            do_sample=True,
            repetition_penalty=1.15,
            no_repeat_ngram_size=3,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            use_cache=True,
        )
    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

print("=" * 70)
print("🧪 EĞİTİM SONRASI UniSense TESTİ")
print("=" * 70)
for q in test_questions:
    print(f"\n❓ {q}")
    print(f"💬 {generate_answer(q)[:500]}")
    print("-" * 60)

### Adım 7: LoRA Merge + GGUF Q4_K_M (Ollama için)

In [ ]:
!pip install -U torchao peft --quiet

import gc, torch, os, shutil
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

del model
gc.collect(); torch.cuda.empty_cache()

base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="auto"
)
merged = PeftModel.from_pretrained(base, f"{UPLOAD_DIR}/lora_unisense_model")
merged = merged.merge_and_unload()
merged.save_pretrained("/kaggle/working/unisense-tr-merged", safe_serialization=True)
tokenizer.save_pretrained("/kaggle/working/unisense-tr-merged")
print("✅ Merge tamamlandı: /kaggle/working/unisense-tr-merged")

del base, merged
gc.collect(); torch.cuda.empty_cache()

!git clone --depth 1 https://github.com/ggerganov/llama.cpp /kaggle/working/llama.cpp 2>/dev/null
!pip install -q -r /kaggle/working/llama.cpp/requirements.txt
!python /kaggle/working/llama.cpp/convert_hf_to_gguf.py \
    /kaggle/working/unisense-tr-merged \
    --outfile /kaggle/working/unisense-tr-f16.gguf --outtype f16

shutil.rmtree("/kaggle/working/unisense-tr-merged", ignore_errors=True)
print("🧹 HF model silindi")

!cd /kaggle/working/llama.cpp && cmake -B build -DGGML_CUDA=OFF && cmake --build build --config Release -j
!/kaggle/working/llama.cpp/build/bin/llama-quantize \
    /kaggle/working/unisense-tr-f16.gguf \
    /kaggle/working/unisense-tr-q4_k_m.gguf Q4_K_M

if os.path.exists("/kaggle/working/unisense-tr-f16.gguf"):
    os.remove("/kaggle/working/unisense-tr-f16.gguf")
    print("🧹 f16 silindi")
shutil.rmtree("/kaggle/working/llama.cpp", ignore_errors=True)
print("🧹 llama.cpp silindi")

size_mb = os.path.getsize("/kaggle/working/unisense-tr-q4_k_m.gguf") / 1024**2
print(f"\n✅ GGUF hazır: unisense-tr-q4_k_m.gguf ({size_mb:.0f} MB)")
!df -h /kaggle/working

In [ ]:
import os, json, subprocess, shutil

GGUF_UPLOAD_DIR = "/kaggle/working/upload-unisense-gguf"
os.makedirs(GGUF_UPLOAD_DIR, exist_ok=True)
shutil.copy2("/kaggle/working/unisense-tr-q4_k_m.gguf", f"{GGUF_UPLOAD_DIR}/unisense-tr-q4_k_m.gguf")

# ⚙️ Sampling parametreleri OPTIMIZE EDİLDİ — Türkçe karakter + repetition loop önler
modelfile = '''FROM ./unisense-tr-q4_k_m.gguf

PARAMETER temperature 0.5
PARAMETER top_p 0.9
PARAMETER top_k 40
PARAMETER min_p 0.05
PARAMETER num_ctx 4096
PARAMETER num_predict 400
PARAMETER repeat_penalty 1.18
PARAMETER repeat_last_n 256
PARAMETER stop "<|im_end|>"
PARAMETER stop "<|im_start|>"

SYSTEM """Sen UniSense — Türkiye 2025 YKS üniversite tercih asistanısın.
YÖK Atlas/ÖSYM verilerine dayanarak Türkçe, kısa, madde işaretli, sayısal cevaplar ver.
ASLA İngilizce kelime kullanma. Bilgi yoksa 'YÖK Atlas'tan kontrol et' de.
Program kodunu [203910363] formatında yaz.
Cevabını maksimum 5 madde ile bitir. Tekrar etme."""
'''
with open(f"{GGUF_UPLOAD_DIR}/Modelfile", "w") as f:
    f.write(modelfile)

with open(f"{GGUF_UPLOAD_DIR}/dataset-metadata.json", "w") as f:
    json.dump({
        "title": "unisense-tr-gguf",
        "id": "ibrahimaskeroglu/unisense-tr-gguf",
        "licenses": [{"name": "CC0-1.0"}]
    }, f)

print("⬆️ GGUF Kaggle'a yükleniyor...")
r = subprocess.run(
    ["kaggle", "datasets", "create", "-p", GGUF_UPLOAD_DIR, "--dir-mode", "zip"],
    capture_output=True, text=True
)
if "already exists" in (r.stdout + r.stderr).lower():
    r = subprocess.run(
        ["kaggle", "datasets", "version", "-p", GGUF_UPLOAD_DIR, "-m", "UniSense Q4_K_M v2 (sampling fix)", "--dir-mode", "zip"],
        capture_output=True, text=True
    )
print(r.stdout[-300:]); print(r.stderr[-300:])

print("\n📦 Lokal makinende:")
print("   kaggle datasets download -d ibrahimaskeroglu/unisense-tr-gguf -p ./unisense-tr-gguf --unzip")
print("   cd unisense-tr-gguf")
print("   ollama create unisense-local -f Modelfile")
print("   ollama run unisense-local")